# BERT rerun — full Ido→Esperanto dictionary expansion

Runs the whole chain in one GPU session:
**finetune → extract → align → train cross-encoder → apply**.

Use a **T4 (or better) GPU** runtime: Runtime → Change runtime type → T4 GPU.
Step 1 (finetune) is the long pole (~2–5 h); the rest is minutes–~30 min.

See `RUN_PLAN_bert_rerun.md` in the repo for the full rationale and the
post-GPU steps (convert + relax the cognate guard + gate).

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + install deps

In [ ]:
!git clone --depth=1 https://github.com/komapc/embedding-aligner.git
%cd embedding-aligner
!pip install -q -r requirements.txt

## 3. Fetch inputs not in the clone

Only the finetune corpus and the cross-encoder positives (bilingual_raw +
langlinks) live as release assets — everything else is in the repo, and the
finetuned model is produced here in step 4.

In [ ]:
REL = 'https://github.com/komapc/embedding-aligner/releases/download/cross-encoder-inputs'
!wget -q {REL}/ido_finetune_corpus.tar.gz
!mkdir -p data/processed && tar xzf ido_finetune_corpus.tar.gz -C data/processed
!wget -q {REL}/cross_encoder_inputs.tar.gz
!mkdir -p extractor_work && tar xzf cross_encoder_inputs.tar.gz -C extractor_work --strip-components=1
!ls -lh data/processed/ido_finetune_corpus.txt extractor_work/

## 4. (1) Finetune XLM-RoBERTa on the Ido corpus  — longest step

Watch for `Corpus loaded: ~432,000 sentences`, then a progress bar with a
decreasing loss. Tuned for a 15 GB T4: `--batch-size 8` (effective 16 via the
hardcoded ×2 accumulation) + `--max-length 128` + expandable_segments avoid
the CUDA OOM that batch-16/seq-256 hits. For a fast first pass use `--epochs 1`.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/13_finetune_bert.py \
  --corpus data/processed/ido_finetune_corpus.txt \
  --output models/bert-ido-finetuned-full \
  --epochs 3 --batch-size 8 --max-length 128

## 5. (2) Extract Ido embeddings from the finetuned model  — ~20–30 min

Uses the committed filtered vocab (32,163). To instead ship the refreshed
vocab, swap in `data/ido_vocabulary_filtered_refreshed.txt` (32,419, mostly
derivable participles — see RUN_PLAN open questions).

In [ ]:
!python3 scripts/14_extract_bert_embeddings.py \
  --model models/bert-ido-finetuned-full \
  --vocab data/ido_vocabulary_filtered.txt \
  --output embeddings/ido_bert_filtered.npz

## 6. (3) Align → new candidates  (GPU embeds EO vocab, then CPU Procrustes/NN)

In [ ]:
!python3 scripts/15_bert_crosslingual_alignment.py \
  --bert-model models/bert-ido-finetuned-full \
  --ido-embeddings embeddings/ido_bert_filtered.npz \
  --epo-vocab models/esperanto_clean__vocab.txt \
  --output-dir results/bert_ido_epo_alignment

## 7. (4) Train the cross-encoder on the NEW candidates  — ~25 min

**Note the final held-out metrics** (F1 / AUC / precision) — they set the
apply threshold in step 8. If a T4 OOMs here, add `--batch-size 16`.

In [ ]:
!python3 scripts/16_train_cross_encoder.py \
  --bilingual-raw extractor_work/bilingual_raw.json \
  --langlinks extractor_work/io_eo_langlinks.json \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --eo-vocab data/esperanto_vocabulary.txt \
  --model-out models/cross-encoder-io-eo --epochs 3 --batch-size 32

## 8. (5) Apply the cross-encoder → filtered pairs  — minutes

Tune `--threshold` from step 7's precision: higher = fewer, cleaner pairs.

In [ ]:
!python3 scripts/17_apply_cross_encoder.py \
  --model models/cross-encoder-io-eo \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --output results/bert_ido_epo_alignment/translation_candidates_ce.json \
  --threshold 0.5 --top-k 3

## 9. Download what the laptop needs

The cross-encoder model + both candidate files. (The ~1 GB finetuned model is
only needed if you later re-extract/re-align — skip unless you want it.)

In [ ]:
!tar czf bert_rerun_outputs.tar.gz \
  results/bert_ido_epo_alignment/translation_candidates.json \
  results/bert_ido_epo_alignment/translation_candidates_ce.json \
  models/cross-encoder-io-eo
from google.colab import files
files.download('bert_rerun_outputs.tar.gz')